# T08 — Feature Engineering

This notebook implements deterministic observation-date feature engineering after the T06 chronological split and before the T07 Train-fitted preprocessing stage. All target-based summaries use **Train only**. No model is trained.

In [1]:
from pathlib import Path
import hashlib
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from fdm_rainfall.data import load_weather_data, chronological_train_validation_test_split
from fdm_rainfall.features import (
    DATE_CANDIDATE_FEATURES, DIFFERENCE_FEATURES, WIND_CYCLICAL_FEATURES,
    WIND_MISSING_INDICATORS, WeatherFeatureEngineer, compass_mapping_table,
)
from fdm_rainfall.preprocessing import (
    ENGINEERED_NUMERICAL_PREDICTORS, STRUCTURAL_NUMERICAL_PREDICTORS,
    fit_transform_engineered_chronological_splits,
)

DATASET = PROJECT_ROOT / 'data/raw/weatherAUS.csv'
TABLE_DIR = PROJECT_ROOT / 'reports/tables'
FIGURE_DIR = PROJECT_ROOT / 'reports/figures/feature_engineering'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

raw_hash = hashlib.sha256(DATASET.read_bytes()).hexdigest().upper()
assert raw_hash == '573FD715CD69FCACC4DF32024D823B450AE3EDAAE7E8FF2EEB623ADBED424014'
raw = load_weather_data(DATASET)
split = chronological_train_validation_test_split(raw)
assert [len(part) for part in split.frames.values()] == [99_546, 21_342, 21_305]
split.summary

,Split,First date,Last date,Rows,Percentage of labelled rows,Unique dates,Locations,RainTomorrow No count,RainTomorrow Yes count,Yes percentage,Missing target count
0,Train,2007-11-01,2015-01-12,99546,70.007666,2541,49,77087,22459,22.561429,0
1,Validation,2015-01-13,2016-04-08,21342,15.009178,452,48,16990,4352,20.391716,0
2,Test,2016-04-09,2017-06-25,21305,14.983157,443,49,16239,5066,23.778456,0


## Candidate generation and Train-only assessment

Positive changes mean afternoon minus morning. Missing wind direction is represented by the neutral point `(0, 0)` plus an explicit missing flag, never by a real compass bearing.

In [2]:
raw_inputs = {name: part.drop(columns=['RainTomorrow']) for name, part in split.frames.items()}
candidate_engineer = WeatherFeatureEngineer(output='candidates').fit(raw_inputs['Train'])
default_engineer = WeatherFeatureEngineer(output='default').fit(raw_inputs['Train'])
candidates = {name: candidate_engineer.transform(frame) for name, frame in raw_inputs.items()}
defaults = {name: default_engineer.transform(frame) for name, frame in raw_inputs.items()}
assert all(len(defaults[name]) == len(split.frames[name]) for name in split.frames)
assert all(defaults[name].index.equals(split.frames[name].index) for name in split.frames)

train_candidates = candidates['Train']
train_target = split.train['RainTomorrow']
assessment_features = [
    'Year', 'Month', 'Month_sin', 'Month_cos', *DIFFERENCE_FEATURES,
    *WIND_CYCLICAL_FEATURES, *WIND_MISSING_INDICATORS, 'Rainfall_log1p',
]
train_assessment = pd.concat([train_candidates[assessment_features], train_target], axis=1)
target_rows = []
for feature in assessment_features:
    for target_value, values in train_assessment.groupby('RainTomorrow')[feature]:
        target_rows.append({
            'Feature': feature, 'RainTomorrow': target_value,
            'Observed count': int(values.notna().sum()),
            'Missing count': int(values.isna().sum()),
            'Mean': float(values.mean()), 'Median': float(values.median()),
            'Q1': float(values.quantile(0.25)), 'Q3': float(values.quantile(0.75)),
        })
train_target_summary = pd.DataFrame(target_rows)
train_target_summary.to_csv(TABLE_DIR / '08_train_engineered_target_summary.csv', index=False)
train_target_summary.head()

,Feature,RainTomorrow,Observed count,Missing count,Mean,Median,Q1,Q3
0,Year,No,77087,0,2011.469664,2011.0,2010.000000,2013.0
1,Year,Yes,22459,0,2011.395120,2011.0,2010.000000,2013.0
2,Month,No,77087,0,6.560523,7.0,4.000000,10.0
3,Month,Yes,22459,0,6.649094,7.0,4.000000,9.0
4,Month_sin,No,77087,0,-0.030508,0.0,-0.866025,0.5


In [3]:
def target_finding(feature):
    rows = train_target_summary.loc[train_target_summary['Feature'].eq(feature)].set_index('RainTomorrow')
    return f"Train mean No={rows.loc['No', 'Mean']:.3f}; Yes={rows.loc['Yes', 'Mean']:.3f}."

inventory_rows = []
def add(feature, source, kind, formula, missing, rationale, finding, redundancy, leakage, decision, reason, preprocessing):
    inventory_rows.append({
        'Feature': feature, 'Source columns': source, 'Type': kind,
        'Formula / mapping': formula, 'Missing-value behavior': missing,
        'Rationale': rationale, 'Train-only descriptive finding': finding,
        'Redundancy concern': redundancy, 'Leakage concern': leakage,
        'Decision': decision, 'Reason': reason,
        'Preprocessing requirement': preprocessing,
        'Retained in default engineered schema?': decision == 'KEEP',
    })

add('Year','Date','Temporal','calendar year','none','Candidate temporal drift signal',target_finding('Year'),'May proxy station coverage or measurement drift','Observation date only','OPTIONAL','Known at prediction time but potentially unstable drift proxy','Train-fitted median/scale if used')
add('Month','Date','Temporal','calendar month 1-12','none','Interpretable seasonality',target_finding('Month'),'Redundant with cyclical month and Season','Observation date only','DROP','Reporting-only; numeric endpoints misrepresent December/January proximity','Not in default matrix')
add('Season','Date','Categorical temporal','Australian mapping: DJF/MAM/JJA/SON','none','Interpretable Australian seasons','Four Train categories','Coarser duplicate of month cycle','Observation date only','DROP','Reporting-only; cyclical month retains smoother detail','Not in default matrix')
add('Month_sin','Date','Cyclical temporal','sin(2π(Month-1)/12)','none','Continuous annual seasonality',target_finding('Month_sin'),'Paired with Month_cos by design','Observation date only','KEEP','Preserves December/January closeness','Median imputation/optional scaling')
add('Month_cos','Date','Cyclical temporal','cos(2π(Month-1)/12)','none','Continuous annual seasonality',target_finding('Month_cos'),'Paired with Month_sin by design','Observation date only','KEEP','Preserves December/January closeness','Median imputation/optional scaling')
difference_specs = {
 'TempRange':('MaxTemp, MinTemp','MaxTemp - MinTemp','Daily thermal range'),
 'TempChange':('Temp3pm, Temp9am','Temp3pm - Temp9am','Positive means warmer at 3pm'),
 'HumidityChange':('Humidity3pm, Humidity9am','Humidity3pm - Humidity9am','Positive means more humid at 3pm'),
 'PressureChange':('Pressure3pm, Pressure9am','Pressure3pm - Pressure9am','Negative means lower pressure at 3pm'),
 'WindSpeedChange':('WindSpeed3pm, WindSpeed9am','WindSpeed3pm - WindSpeed9am','Positive means stronger 3pm wind'),
}
for feature,(source,formula,rationale) in difference_specs.items():
    add(feature,source,'Numerical difference',formula,'missing if either source is missing',rationale,target_finding(feature),'Deterministic combination; originals retained','Same-day measurements only','KEEP','Meaningful within-day change without discarding source levels','Train-fitted median/optional scaling')
for source in ('WindGustDir','WindDir9am','WindDir3pm'):
    for component in ('sin','cos'):
        feature=f'{source}_{component}'
        add(feature,source,'Cyclical wind',f'{component}(direction degrees)','missing direction -> 0 plus explicit indicator','Preserves circular compass geometry',target_finding(feature),'Replaces, rather than duplicates, direction one-hot','Same-day direction only','KEEP','Compact circular default; one-hot remains T07 baseline alternative','Median imputation/optional scaling')
    indicator=f'{source}_missing'
    add(indicator,source,'Binary indicator','1 when original direction missing else 0','never missing','Distinguishes neutral encoding from observed direction',target_finding(indicator),'Required companion to circular encoding','Same-day missingness only','KEEP','Prevents silent assignment of missing to a bearing','Binary passthrough; not scaled')
for source in STRUCTURAL_NUMERICAL_PREDICTORS:
    feature=f'{source}_missing'
    missing_count=int(split.train[source].isna().sum())
    add(feature,source,'Binary indicator','created by T07 before imputation','never missing','Retains station-dependent structural missingness',f'{missing_count:,} Train source values missing.','Existing T07 feature; do not duplicate','Same-day missingness only','KEEP','T03/T07 evidence supports retaining the missingness signal','Created once by preprocessing; not scaled')
add('Rainfall_log1p','Rainfall','Monotonic transform','log1p(Rainfall)','missing when Rainfall missing','Compresses strong right skew',target_finding('Rainfall_log1p'),'Highly related to raw Rainfall','Same-day rainfall only','OPTIONAL','May help linear/scale-sensitive models; tree models may not need it','Train-fitted median/optional scaling if enabled')
add('ClimateZone','Location','Proposed categorical','authoritative mapping not available','not implemented','Could summarize climate geography','Not assessed','Would overlap Location','Unsupported mapping could encode assumptions','DEFERRED','No authoritative mapping is incorporated','None until defensible mapping exists')

inventory = pd.DataFrame(inventory_rows)
inventory.to_csv(TABLE_DIR / '08_engineered_feature_inventory.csv', index=False)
inventory[['Feature','Decision','Retained in default engineered schema?','Reason']].to_csv(TABLE_DIR / '08_feature_decisions.csv', index=False)
inventory['Decision'].value_counts()

Decision
KEEP        20
OPTIONAL     2
DROP         2
DEFERRED     1
Name: count, dtype: int64

In [4]:
created_features = [*DATE_CANDIDATE_FEATURES, *DIFFERENCE_FEATURES, *WIND_CYCLICAL_FEATURES, *WIND_MISSING_INDICATORS, 'Rainfall_log1p']
missing_rows = []
for split_name, frame in candidates.items():
    for feature in created_features:
        missing_rows.append({
            'Split': split_name, 'Feature': feature, 'Rows': len(frame),
            'Missing count': int(frame[feature].isna().sum()),
            'Missing percentage': float(frame[feature].isna().mean()*100),
            'Handling': 'T07 Train-fitted imputation' if frame[feature].isna().any() else 'No imputation required',
        })
engineered_missingness = pd.DataFrame(missing_rows)
engineered_missingness.to_csv(TABLE_DIR / '08_engineered_missingness.csv', index=False)

redundancy_specs = {
 'TempRange':['MinTemp','MaxTemp'], 'TempChange':['Temp9am','Temp3pm'],
 'HumidityChange':['Humidity9am','Humidity3pm'], 'PressureChange':['Pressure9am','Pressure3pm'],
 'WindSpeedChange':['WindSpeed9am','WindSpeed3pm'], 'Rainfall_log1p':['Rainfall'],
}
redundancy_rows=[]
for engineered, sources in redundancy_specs.items():
    for source in sources:
        redundancy_rows.append({'Engineered feature':engineered,'Compared with':source,'Train Pearson correlation':float(train_candidates[[engineered,source]].corr().iloc[0,1]),'Decision':'Keep both for now' if engineered!='Rainfall_log1p' else 'Rainfall_log1p optional','Reason':'Difference preserves movement/range while source preserves level' if engineered!='Rainfall_log1p' else 'Monotonic transform is algorithm-dependent'})
redundancy_rows.extend([
 {'Engineered feature':'Month_sin + Month_cos','Compared with':'Month + Season','Train Pearson correlation':np.nan,'Decision':'Keep cyclical pair; drop Month/Season from default','Reason':'Avoid duplicate seasonal representations while preserving circular proximity'},
 {'Engineered feature':'Wind direction sine/cosine','Compared with':'Wind direction one-hot','Train Pearson correlation':np.nan,'Decision':'Use cyclical pair in engineered default','Reason':'Compact circular geometry; T07 one-hot remains a baseline alternative'},
])
redundancy = pd.DataFrame(redundancy_rows)
redundancy.to_csv(TABLE_DIR / '08_feature_redundancy_summary.csv', index=False)

wind_mapping = compass_mapping_table()
wind_mapping.to_csv(TABLE_DIR / '08_cyclical_wind_mapping.csv', index=False)
date_summary = pd.DataFrame([
 {'Feature':'Year','Train minimum':int(train_candidates.Year.min()),'Train maximum':int(train_candidates.Year.max()),'Unique values':int(train_candidates.Year.nunique()),'Decision':'OPTIONAL','Reason':'Known at prediction time but may proxy drift/coverage'},
 {'Feature':'Month','Train minimum':1,'Train maximum':12,'Unique values':12,'Decision':'DROP','Reason':'Reporting only; redundant with cyclical pair'},
 {'Feature':'Season','Train minimum':'Autumn','Train maximum':'Winter','Unique values':4,'Decision':'DROP','Reason':'Reporting only; coarser redundant seasonality'},
 {'Feature':'Month_sin','Train minimum':float(train_candidates.Month_sin.min()),'Train maximum':float(train_candidates.Month_sin.max()),'Unique values':12,'Decision':'KEEP','Reason':'Circular annual coordinate'},
 {'Feature':'Month_cos','Train minimum':float(train_candidates.Month_cos.min()),'Train maximum':float(train_candidates.Month_cos.max()),'Unique values':12,'Decision':'KEEP','Reason':'Circular annual coordinate'},
])
date_summary.to_csv(TABLE_DIR / '08_date_feature_summary.csv', index=False)
rainfall_summary = pd.DataFrame([
 {'Representation':'Rainfall','Observed Train rows':int(train_candidates.Rainfall.notna().sum()),'Missing Train rows':int(train_candidates.Rainfall.isna().sum()),'Skewness':float(train_candidates.Rainfall.skew()),'Q99':float(train_candidates.Rainfall.quantile(.99)),'Maximum':float(train_candidates.Rainfall.max()),'Decision':'KEEP'},
 {'Representation':'Rainfall_log1p','Observed Train rows':int(train_candidates.Rainfall_log1p.notna().sum()),'Missing Train rows':int(train_candidates.Rainfall_log1p.isna().sum()),'Skewness':float(train_candidates.Rainfall_log1p.skew()),'Q99':float(train_candidates.Rainfall_log1p.quantile(.99)),'Maximum':float(train_candidates.Rainfall_log1p.max()),'Decision':'OPTIONAL'},
])
rainfall_summary.to_csv(TABLE_DIR / '08_rainfall_transform_summary.csv', index=False)
rainfall_summary

,Representation,Observed Train rows,Missing Train rows,Skewness,Q99,Maximum,Decision
0,Rainfall,98539,1007,9.967964,38.000000,371.000000,KEEP
1,Rainfall_log1p,98539,1007,2.033358,3.663562,5.918894,OPTIONAL


## Integration: raw split → feature engineering → Train-fitted preprocessing

The original T07 one-hot configuration remains available. This engineered configuration removes raw wind-direction categories from its default input and uses circular coordinates with explicit missing flags.

In [5]:
processed = fit_transform_engineered_chronological_splits(split, scale_numeric=False)
processed_scaled = fit_transform_engineered_chronological_splits(split, scale_numeric=True)
processed_frames = {'Train':processed.X_train,'Validation':processed.X_validation,'Test':processed.X_test}
scaled_frames = {'Train':processed_scaled.X_train,'Validation':processed_scaled.X_validation,'Test':processed_scaled.X_test}

schema_rows=[]
for feature in processed.preprocessor.get_feature_names_out():
    if feature in ENGINEERED_NUMERICAL_PREDICTORS:
        role='Numerical/cyclical predictor'; source='Raw measurement or T08 deterministic formula'; requirement='Train-fitted imputation; optional scaling'
    elif feature.endswith('_missing'):
        role='Missing indicator'; source='Original missingness'; requirement='Binary passthrough; not scaled'
    else:
        role='One-hot categorical column'; source='Location or RainToday'; requirement='Train-fitted one-hot; not scaled'
    schema_rows.append({'Processed feature':feature,'Source':source,'Role':role,'Data type':'float64','Missing-value behavior':'No missing values after preprocessing','Leakage risk':'None: deterministic or Train-fitted only','Preprocessing requirement':requirement})
engineered_schema = pd.DataFrame(schema_rows)
engineered_schema.to_csv(TABLE_DIR / '08_engineered_schema_summary.csv', index=False)

expected_rows={'Train':99_546,'Validation':21_342,'Test':21_305}
leakage_checks=[
 ('RainTomorrow excluded from engineered predictors','RainTomorrow' not in processed.X_train.columns,'Target is separated before engineering/preprocessing'),
 ('RISK_MM absent',all('RISK_MM' not in part.columns for part in split.frames.values()),'Raw/split schema contains no RISK_MM'),
 ('Date not directly encoded or scaled','Date' not in processed.X_train.columns,'Only observation-date month cycle is retained'),
 ('No tomorrow/future weather columns used',not any('Tomorrow' in c for c in processed.X_train.columns),'All formulas use observation-date sources'),
 ('Feature engineer learns no target parameters',not hasattr(processed.feature_engineer,'target_'),'Transformer fit validates schema only'),
 ('Preprocessor fitted on Train rows only',processed.preprocessor.fit_row_count_==99_546,'Validation/Test use unchanged Train parameters'),
 ('No Validation/Test target used for decisions',True,'Only train_target_summary was calculated'),
 ('Train rows preserved',len(processed.X_train)==99_546,'No row filtering'),
 ('Validation rows preserved',len(processed.X_validation)==21_342,'No row filtering'),
 ('Test rows preserved',len(processed.X_test)==21_305,'No row filtering'),
 ('Target indices aligned',all(processed_frames[n].index.equals(getattr(processed,f"y_{n.lower()}").index) for n in processed_frames),'Predictor and target indices match'),
 ('Processed outputs finite and complete',all(not x.isna().any().any() and not np.isinf(x.to_numpy()).any() for x in [*processed_frames.values(),*scaled_frames.values()]),'Both unscaled and scaled paths checked'),
]
leakage_table=pd.DataFrame([{'Check':c,'Passed':bool(p),'Evidence':e,'Result':'PASS' if p else 'FAIL'} for c,p,e in leakage_checks])
leakage_table.to_csv(TABLE_DIR / '08_feature_engineering_leakage_checks.csv', index=False)
assert leakage_table.Result.eq('PASS').all()

feature_counts=pd.DataFrame([
 {'Stage':'Before T08','Measure':'Original raw non-Date predictors','Count':21,'Explanation':'16 numerical + 5 categorical'},
 {'Stage':'Before T08','Measure':'T07 processed features','Count':124,'Explanation':'16 numerical + 4 structural indicators + 104 one-hot'},
 {'Stage':'T08 candidates','Measure':'Implemented engineered raw features','Count':20,'Explanation':'5 date + 5 differences + 6 wind coordinates + 3 wind indicators + log1p rainfall'},
 {'Stage':'T08 decisions','Measure':'New T08 features retained by default','Count':16,'Explanation':'2 month-cycle + 5 differences + 6 wind coordinates + 3 wind indicators'},
 {'Stage':'T08 decisions','Measure':'Optional implemented features','Count':2,'Explanation':'Year and Rainfall_log1p'},
 {'Stage':'T08 decisions','Measure':'Dropped implemented representations','Count':2,'Explanation':'Month and Season retained for reporting only'},
 {'Stage':'T08 decisions','Measure':'Deferred proposal features','Count':1,'Explanation':'ClimateZone lacks an authoritative mapping'},
 {'Stage':'Engineered default','Measure':'Raw inputs to preprocessing','Count':34,'Explanation':'18 retained originals + 16 new T08 defaults'},
 {'Stage':'Engineered default','Measure':'Processed numerical columns','Count':29,'Explanation':'16 originals + 2 month-cycle + 5 differences + 6 wind coordinates'},
 {'Stage':'Engineered default','Measure':'Processed indicator columns','Count':7,'Explanation':'4 T07 structural + 3 T08 wind-direction indicators'},
 {'Stage':'Engineered default','Measure':'Processed one-hot columns','Count':53,'Explanation':'50 Location + 3 RainToday'},
 {'Stage':'Engineered default','Measure':'Final processed features','Count':89,'Explanation':'29 numerical + 7 indicators + 53 one-hot'},
])
feature_counts.to_csv(TABLE_DIR / '08_final_feature_counts.csv', index=False)
feature_counts

,Stage,Measure,Count,Explanation
0,Before T08,Original raw non-Date predictors,21,16 numerical + 5 categorical
1,Before T08,T07 processed features,124,16 numerical + 4 structural indicators + 104 o...
2,T08 candidates,Implemented engineered raw features,20,5 date + 5 differences + 6 wind coordinates + ...
3,T08 decisions,New T08 features retained by default,16,2 month-cycle + 5 differences + 6 wind coordin...
4,T08 decisions,Optional implemented features,2,Year and Rainfall_log1p
5,T08 decisions,Dropped implemented representations,2,Month and Season retained for reporting only
6,T08 decisions,Deferred proposal features,1,ClimateZone lacks an authoritative mapping
7,Engineered default,Raw inputs to preprocessing,34,18 retained originals + 16 new T08 defaults
8,Engineered default,Processed numerical columns,29,16 originals + 2 month-cycle + 5 differences +...
9,Engineered default,Processed indicator columns,7,4 T07 structural + 3 T08 wind-direction indica...


In [6]:
plot_frame=pd.concat([train_candidates[list(DIFFERENCE_FEATURES)],train_target],axis=1)
fig,axes=plt.subplots(2,3,figsize=(15,8))
for ax,feature in zip(axes.flat,DIFFERENCE_FEATURES):
    groups=[plot_frame.loc[plot_frame.RainTomorrow.eq(label),feature].dropna() for label in ('No','Yes')]
    ax.boxplot(groups,tick_labels=['No','Yes'],showfliers=False)
    ax.set_title(feature); ax.set_xlabel('RainTomorrow'); ax.set_ylabel(feature)
axes.flat[-1].axis('off')
fig.suptitle('Train-only weather-difference distributions by RainTomorrow')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'08_weather_difference_target_distributions.png',dpi=160,bbox_inches='tight'); plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(12,4.5))
raw_limit=train_candidates.Rainfall.quantile(.995)
axes[0].hist(train_candidates.Rainfall.clip(upper=raw_limit).dropna(),bins=60,color='#3274a1')
axes[0].set_title('Raw Rainfall (clipped at Train 99.5th percentile)'); axes[0].set_xlabel('mm')
axes[1].hist(train_candidates.Rainfall_log1p.dropna(),bins=60,color='#e1812c')
axes[1].set_title('log1p(Rainfall), full Train range'); axes[1].set_xlabel('log1p(mm)')
fig.suptitle('Train-only rainfall representation comparison')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'08_rainfall_raw_vs_log1p.png',dpi=160,bbox_inches='tight'); plt.close(fig)

fig,axes=plt.subplots(1,2,figsize=(12,5))
months=np.arange(1,13); angles=2*np.pi*(months-1)/12
axes[0].scatter(np.cos(angles),np.sin(angles),c=months,cmap='twilight')
for m,x,y in zip(months,np.cos(angles),np.sin(angles)): axes[0].annotate(str(m),(x,y))
axes[0].set_title('Cyclical month representation')
axes[1].scatter(wind_mapping.Cosine,wind_mapping.Sine,color='#55a868')
for _,r in wind_mapping.iterrows(): axes[1].annotate(r.Direction,(r.Cosine,r.Sine))
axes[1].set_title('Compass-direction representation')
for ax in axes: ax.set_aspect('equal'); ax.axhline(0,color='grey',lw=.5); ax.axvline(0,color='grey',lw=.5); ax.set_xlabel('cos'); ax.set_ylabel('sin')
fig.tight_layout(); fig.savefig(FIGURE_DIR/'08_cyclical_encodings.png',dpi=160,bbox_inches='tight'); plt.close(fig)
print('Created 11 T08 tables and 3 focused figures.')

Created 11 T08 tables and 3 focused figures.


In [7]:
post_counts=[]
for name,frame in processed_frames.items():
    post_counts.append({'Split':name,'Rows':len(frame),'Features':frame.shape[1],'NaN':int(frame.isna().sum().sum()),'None':int(frame.isnull().sum().sum()),'Infinite':int(np.isinf(frame.to_numpy(dtype=float)).sum()),'Target aligned':frame.index.equals(getattr(processed,f"y_{name.lower()}").index),'Date aligned':frame.index.equals(getattr(processed,f"dates_{name.lower()}").index)})
post_counts=pd.DataFrame(post_counts)
assert (post_counts[['NaN','None','Infinite']].to_numpy()==0).all()
assert post_counts['Target aligned'].all() and post_counts['Date aligned'].all()
print(f'Raw checksum: {raw_hash}')
print('Default raw engineered schema:', defaults['Train'].shape[1], 'features')
print('Final processed engineered schema:', processed.X_train.shape[1], 'features')
post_counts

Raw checksum: 573FD715CD69FCACC4DF32024D823B450AE3EDAAE7E8FF2EEB623ADBED424014
Default raw engineered schema: 34 features
Final processed engineered schema: 89 features


,Split,Rows,Features,NaN,None,Infinite,Target aligned,Date aligned
0,Train,99546,89,0,0,0,True,True
1,Validation,21342,89,0,0,0,True,True
2,Test,21305,89,0,0,0,True,True
